# Unit 6: Building a Reproducible GeoAI Pipeline

This notebook supports the Unit 6 ILOs:

- construct a complete GeoAI workflow from preprocessing to evaluation,
- apply an AI method to a new hazard mapping problem.

The focus here is workflow design. The notebook ties together preprocessing, feature engineering, model training, prediction, and evaluation in one reusable structure.

## Why Reproducibility Matters

A good GeoAI workflow should be understandable, repeatable, and adaptable. In hazard mapping, results may support scientific conclusions or practical decisions, so it is important that another student or researcher can follow the same workflow and understand how the final output was produced.

This notebook is therefore organized as a pipeline rather than a one-off script. Each stage has a clear responsibility: loading data, creating features, splitting the dataset, building a model, and evaluating the result. That structure makes future modification much easier.

## 1. Pipeline design principles

A reproducible GeoAI workflow should make data sources, preprocessing assumptions, model configuration, and evaluation choices explicit. Even if a model changes later, the overall pipeline contract should remain stable.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report
except ImportError:
    RandomForestClassifier = None
    train_test_split = None
    classification_report = None

ROOT = Path('data/unit6_pipeline')
ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
@dataclass
class PipelineConfig:
    hazard_name: str = 'new_hazard'
    random_state: int = 42
    test_size: float = 0.2
    n_estimators: int = 200
    max_depth: int | None = 12
    feature_names: tuple = ('blue', 'green', 'red', 'nir', 'ndvi', 'slope')

config = PipelineConfig()
config

In [ ]:
def load_or_create_dataset(config):
    rng = np.random.default_rng(config.random_state)
    n = 4000
    blue = rng.uniform(0, 1, n)
    green = rng.uniform(0, 1, n)
    red = rng.uniform(0, 1, n)
    nir = rng.uniform(0, 1, n)
    ndvi = (nir - red) / (nir + red + 1e-6)
    slope = rng.uniform(0, 45, n)
    hazard = ((ndvi < 0.1) & (slope > 20)).astype(int)
    hazard = np.where(rng.uniform(0, 1, n) < 0.08, 1 - hazard, hazard)
    return pd.DataFrame({
        'blue': blue,
        'green': green,
        'red': red,
        'nir': nir,
        'ndvi': ndvi,
        'slope': slope,
        'label': hazard,
    })

df = load_or_create_dataset(config)
df.head()

In [ ]:
def preprocess_features(df, config):
    X = df[list(config.feature_names)].copy()
    y = df['label'].copy()
    return X, y

def split_data(X, y, config):
    return train_test_split(X, y, test_size=config.test_size, random_state=config.random_state, stratify=y)

def build_model(config):
    return RandomForestClassifier(
        n_estimators=config.n_estimators,
        max_depth=config.max_depth,
        random_state=config.random_state,
        n_jobs=-1,
    )

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    return y_pred, pd.DataFrame(report).T

def feature_importance_table(model, feature_names):
    return pd.DataFrame({'feature': feature_names, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)

In [ ]:
X, y = preprocess_features(df, config)

# Uncomment these lines when scikit-learn is available in the course environment.
# X_train, X_test, y_train, y_test = split_data(X, y, config)
# model = build_model(config)
# model.fit(X_train, y_train)
# y_pred, report_df = evaluate_model(model, X_test, y_test)
# importances = feature_importance_table(model, list(config.feature_names))

print('Pipeline scaffold complete. Uncomment the training lines when dependencies are installed.')

## Adapting the pipeline to a new hazard

1. Update the data loader.
2. Choose hazard-relevant features.
3. Replace the model if needed.
4. Reuse the same evaluation contract.
5. Document all assumptions.

This keeps the workflow reproducible, reviewable, and easier to maintain across student projects.

## Unit 6 Takeaway

The central lesson of this notebook is that strong GeoAI practice depends on workflow design as much as model choice. A reusable pipeline makes assumptions explicit, separates steps clearly, and allows new data or models to be inserted without losing clarity.

Students should now be able to see how preprocessing, feature engineering, training, and evaluation fit together as one connected system. That systems view is essential for building robust hazard mapping workflows beyond the classroom.